# Meridian GeoX Demo
This demo showcases the fundamental functionalities and basic usage of the library.

## Setup and Installation
Please follow the steps bellow in order to install and use the Meridian GeoX library using Colab.
1. **Connect to Hosted Runtime**

2. **GitHub Authentication**
Because the meridian_geox repository is currently private, you must provide a GitHub Personal Access Token (PAT) to clone the code and access the simulation datasets.
  
3. **Generate a Token:** Visit https://github.com/settings/tokens and generate a new token with the repo scope enabled.
4. **Security:** The setup cell below uses getpass.getpass() to ensure your token is handled as a secret and is not saved in the notebook's output or history.

In [ ]:
import getpass
import os

# Securely prompt for your GitHub Personal Access Token (PAT)
github_token = getpass.getpass("Enter your GitHub PAT: ")

# Clone the repo locally if it doesn't exist
if not os.path.exists("meridian-geox"):
  !git clone https://{github_token}@github.com/google/meridian-geox.git
else:
  print("Repository already exists. Skipping clone.")

# Install the library from the local folder
!pip install ./meridian-geox

In [ ]:
from pprint import pprint
import warnings
import meridian_geox as geox
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in divide",
    category=RuntimeWarning,
)

pd.set_option("display.max_columns", None)

meridian_geox_root = None

In [ ]:
# If you prefer to save the design in Google Drive, run this cell.
import os
from google.colab import drive

drive_mount = '/content/drive'
drive.mount(drive_mount, force_remount=True)

# Optional: specify a subfolder to organize your runs
subfolder = ''  # @param {"type":"string", "placeholder": "e.g., my_geox_project"}

# Define the root path specifically for GeoX designs and reports
meridian_geox_root = f'{drive_mount}/MyDrive/{subfolder}'

# Create the directory if it doesn't exist
if not os.path.exists(meridian_geox_root):
  os.makedirs(meridian_geox_root)
  print(f'Created directory: {meridian_geox_root}')
else:
  print(f'Using directory: {meridian_geox_root}')

## Single cell design

In [ ]:
# Specify the internal path to the design data CSV in the repository
design_data_path = "/content/meridian-geox/meridian_geox/data/example_design_data_single_cell.csv"

# Load the data and the token provided in cell-1
design_data = pd.read_csv(design_data_path)

# Preview the first few rows to ensure it loaded correctly
design_data.head()

In [ ]:
for colname in ["conversions", "spend"]:
  design_data[colname] = pd.to_numeric(design_data[colname])
design_data["date"] = pd.to_datetime(design_data["date"])
design_data["location"] = design_data["location"].astype(str)
design_data.head()

In [ ]:
# Design experiment
print("\nDesigning experiment...")
single_cell_design_config = geox.DesignConfig(
    experiment_duration=30,
    experiment_types=geox.ExperimentType.HOLDBACK,
    methodology=geox.Methodology.TBR,
    geo_assignment_rule=geox.GeoAssignmentRule.STRATIFIED_SAMPLING,
    cost_per_incremental_conversion=1,
    cell_count=1,
    design_output_count=5,
)
single_cell_constraints = geox.Constraints(
    excluded_geos={"105"},
    budget_constraint=geox.Budget(budget=500000),
    max_conversions_percent=0.3,
)
single_cell_design_set = geox.run_design(
    design_data,
    single_cell_design_config,
    single_cell_constraints,
)

In [ ]:
print("Experiment design generated:")
single_cell_selected_design_id = next(iter(single_cell_design_set.designs))
pprint(single_cell_design_set.designs[single_cell_selected_design_id])

In [ ]:
print("Design metrics:")
single_cell_design_set.design_metrics

In [ ]:
# User saves the string to disk
single_cell_saved_design_json = single_cell_design_set.designs[
    single_cell_selected_design_id
].export_to_json()

In [ ]:
# Write the JSON string to a file on Google Drive if mounted to Google Drive
if meridian_geox_root:
  design_file_path = os.path.join(meridian_geox_root, "single_cell_design.json")
  with open(design_file_path, "w") as f:
    f.write(single_cell_saved_design_json)
  print(f"Design successfully saved to Google Drive at: {design_file_path}")
else:
  print("Google Drive not mounted. Design exists in session memory only.")

In [ ]:
geox.plot_design(single_cell_design_set.designs[single_cell_selected_design_id])

## Single cell analysis

In [ ]:
# Specify the path to the analysis CSV
analysis_data_path = "/content/meridian-geox/meridian_geox/data/example_analysis_data_single_cell_holdback.csv"

# Load the data
analysis_data = pd.read_csv(analysis_data_path)
analysis_data.head()

In [ ]:
for colname in ["conversions", "spend"]:
  analysis_data[colname] = pd.to_numeric(analysis_data[colname])
analysis_data["date"] = pd.to_datetime(analysis_data["date"])
analysis_data["location"] = analysis_data["location"].astype(str)
analysis_data.head()

In [ ]:
# Try to load the design from Google Drive, with a fallback to session memory.
single_cell_loaded_design = None

if meridian_geox_root:
  design_file_path = os.path.join(meridian_geox_root, "single_cell_design.json")
  if os.path.exists(design_file_path):
    with open(design_file_path, "r") as f:
      design_json_text = f.read()
    single_cell_loaded_design = geox.Design.load_from_json(design_json_text)
    print(f"Design successfully loaded from Google Drive: {design_file_path}")
  else:
    print(
        f"Design file not found at {design_file_path}. Will try to load from"
        " session memory."
    )

if single_cell_loaded_design is None:
  try:
    single_cell_loaded_design = geox.Design.load_from_json(
        single_cell_saved_design_json
    )
    print("Design successfully loaded from session memory.")
  except NameError:
    print(
        "Error: Design data not found on Drive or in memory. Please run the"
        " design step first."
    )

In [ ]:
# Analyze experiment results
print("\nAnalyzing experiment results...")
single_cell_analysis_config = geox.AnalysisConfig(
    design=single_cell_loaded_design,
    analysis_start_date=pd.to_datetime("2020-04-01"),
    analysis_end_date=pd.to_datetime("2020-04-30"),
)
single_cell_analysis_result = geox.analyze(
    analysis_data, single_cell_analysis_config
)
print(f"Analysis result:")
pprint(single_cell_analysis_result)

In [ ]:
single_cell_analysis_result.results

In [ ]:
single_cell_analysis_result.results["cell_1"].cumulative_lift.head()

In [ ]:
single_cell_analysis_result.results["cell_1"].cumulative_icpd.head()

In [ ]:
single_cell_analysis_result.results["cell_1"].pointwise_difference.head()

In [ ]:
geox.plot_analysis(single_cell_analysis_result)

## Compare designs using different methodologies

In [ ]:
comparison_design_results = geox.compare_designs(
    design_data,
    [
        (
            geox.DesignConfig(
                experiment_duration=30,
                methodology=geox.Methodology.TBR,
                geo_assignment_rule=geox.GeoAssignmentRule.RANDOM,
                design_output_count=10,
            ),
            geox.Constraints(),
        ),
        (
            geox.DesignConfig(
                experiment_duration=30,
                methodology=geox.Methodology.TBR,
                geo_assignment_rule=geox.GeoAssignmentRule.STRATIFIED_SAMPLING,
            ),
            geox.Constraints(),
        ),
    ],
)

In [ ]:
# Best design (by MDE)
comparison_selected_design_id = next(iter(comparison_design_results.designs))
pprint(comparison_design_results.designs[comparison_selected_design_id])

In [ ]:
comparison_design_results.design_metrics

## Concatenate two designs manually and return top N designs

In [ ]:
# Generate two single-cell designs
print("\nGenerating first single-cell design...")
design_config_1 = geox.DesignConfig(
    experiment_duration=30,
    methodology=geox.Methodology.TBR,
    geo_assignment_rule=geox.GeoAssignmentRule.RANDOM,
    cell_count=1,
)
design_set_1 = geox.run_design(
    design_data,
    design_config_1,
    constraints=geox.Constraints(),
)
print("First single-cell design generated.")
design_set_1.design_metrics

In [ ]:
print("\nGenerating second single-cell design...")
design_config_2 = geox.DesignConfig(
    experiment_duration=30,
    methodology=geox.Methodology.TBR,
    geo_assignment_rule=geox.GeoAssignmentRule.STRATIFIED_SAMPLING,
    cell_count=1,
)
design_set_2 = geox.run_design(
    design_data, design_config_2, constraints=geox.Constraints()
)
print("Second single-cell design generated.")
design_set_2.design_metrics

In [ ]:
# Concatenate the two newly generated design sets
print("\nConcatenating the two single-cell design sets...")
combined_design_set = geox.concat_design_reports([design_set_1, design_set_2])

print("Combined design metrics (ranked by MDE):")
combined_design_set.design_metrics